# Lung Mask Generation Comparison
**HU Threshold (own contribution) vs TotalSegmentator (ORACLE-CT baseline)**

This notebook compares two approaches to generating lung foreground masks on LIDC-IDRI CT slices:
- **HU Threshold**: physics-based, zero annotations, zero pretrained models — the novel contribution of this dissertation
- **TotalSegmentator**: pretrained deep-learning organ segmentor used in ORACLE-CT (Dahal et al. 2026)

No model training occurs here. This is purely a mask quality comparison.

## Cell 1 — Imports and environment check

In [ ]:
%matplotlib inline

import sys
import os
import subprocess

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import ndimage
import torch
import pylidc as pl

# ── Version printout ────────────────────────────────────────────────
print(f"Python      : {sys.version}")
print(f"NumPy       : {np.__version__}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA avail  : {torch.cuda.is_available()}")
print(f"Matplotlib  : {matplotlib.__version__}")

# ── TotalSegmentator — install if missing, then import ──────────────
TS_AVAILABLE = False
try:
    import totalsegmentator
    from totalsegmentator.python_api import totalsegmentator as run_ts
    TS_AVAILABLE = True
    print(f"TotalSegmentator: available (v{totalsegmentator.__version__})")
except ImportError:
    print("TotalSegmentator not found — attempting pip install ...")
    try:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q',
             'TotalSegmentator', 'nibabel'],
            stdout=subprocess.DEVNULL
        )
        import totalsegmentator
        from totalsegmentator.python_api import totalsegmentator as run_ts
        TS_AVAILABLE = True
        print(f"TotalSegmentator: installed and available (v{totalsegmentator.__version__})")
    except Exception as e:
        print(f"TotalSegmentator: install failed ({e})")
        print("  TS masks will be None throughout. Run manually: pip install TotalSegmentator nibabel")

# ── pylidc connectivity check ────────────────────────────────────────
try:
    test_scans = pl.query(pl.Scan).limit(1).all()
    print(f"pylidc      : connected ({len(test_scans)} test scan found)")
except Exception as e:
    print(f"pylidc      : ERROR — {e}")

# ── Config ──────────────────────────────────────────────────────────
SAMPLE_SIZE = 10    # number of nodule slices to compare
MIN_RADS    = 2     # min radiologists per nodule
DILATION_PX = 3     # dilation radius — same as Config.MASK_DILATION
RESULTS_DIR = os.path.join(os.getcwd(), 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"\nResults dir : {RESULTS_DIR}")
print(f"Sample size : {SAMPLE_SIZE}")

## Cell 2 — Load a small sample of LIDC-IDRI nodule slices

We load `SAMPLE_SIZE` scans and for each pick the first nodule with at least 2 radiologist annotations.
Only the raw HU slice and patient metadata are stored — no full 3-D volumes kept in RAM.

In [ ]:
samples = []   # list of dicts: {patient_id, slice_2d, volume, slice_idx}

all_scans = pl.query(pl.Scan).limit(SAMPLE_SIZE * 5).all()
print(f"Querying up to {SAMPLE_SIZE * 5} scans to find {SAMPLE_SIZE} valid samples")

for scan in all_scans:
    if len(samples) >= SAMPLE_SIZE:
        break
    try:
        nodules = scan.cluster_annotations()
        nodule = next(
            (n for n in nodules if len(n) >= MIN_RADS), None
        )
        if nodule is None:
            continue

        volume    = scan.to_volume()   # (H, W, D) int16 HU
        ann       = nodule[0]
        k_indices = sorted(set([c.image_k_position for c in ann.contours]))
        slice_idx = k_indices[len(k_indices) // 2]
        slice_2d  = volume[:, :, slice_idx].astype(np.float32)

        samples.append({
            'patient_id': scan.patient_id,
            'slice_2d'  : slice_2d,
            'volume'    : volume,
            'slice_idx' : slice_idx,
        })
        print(f"  Loaded {scan.patient_id}  slice={slice_idx}  shape={slice_2d.shape}")

    except Exception as e:
        print(f"  Skipping {scan.patient_id}: {e}")
        continue

print(f"\nSuccessfully loaded {len(samples)} samples.")

## Cell 3 — HU Threshold lung mask

**[OWN CONTRIBUTION]** Physics-based HU threshold lung foreground mask.

Imported directly from `dataset.py` — the exact same function used during training.
Threshold: −500 HU (lung air). Border-removal fix prevents background air from being
mistaken for lung. 3-voxel dilation captures juxta-pleural nodules [ORACLE-CT 2026].

In [ ]:
# Add MVP2 root to path so we can import dataset.py
MVP2_ROOT = os.getcwd()
if MVP2_ROOT not in sys.path:
    sys.path.insert(0, MVP2_ROOT)

from dataset import LIDCDataset

# Grab the unbound method — _get_lung_mask uses only Config constants,
# not self, so passing None as self is safe.
_get_lung_mask = LIDCDataset._get_lung_mask

# [OWN CONTRIBUTION] Physics-based HU threshold lung foreground mask
# Novel: no prior work uses HU thresholding as a Grad-CAM attention
# training signal. Literature uses HU only for preprocessing, or uses
# learned segmentation (Zhang et al. 2022, ORACLE-CT 2026)
hu_masks = []
for s in samples:
    mask = _get_lung_mask(None, s['slice_2d'])
    hu_masks.append(mask)
    cov = mask.mean() * 100
    print(f"  {s['patient_id']}  HU mask coverage: {cov:.1f}%")

print(f"\nHU masks computed for {len(hu_masks)} samples.")

## Cell 4 — TotalSegmentator lung mask

**[ORACLE-CT, DAHAL ET AL. 2026]** TotalSegmentator organ mask.
Dahal *"Organ-Aware Attention Improves CT Triage"* arXiv 2026.

TotalSegmentator takes a 3-D NIfTI volume and returns organ segmentation masks.
We extract the lung lobe labels and project onto our target axial slice.
The same 3-voxel dilation is applied for a fair comparison.

If TotalSegmentator is not installed or fails, that sample gets `None` — the
notebook continues and skips that sample in quantitative comparisons.

In [ ]:
import tempfile
import shutil

ts_masks = []   # parallel list to samples; None where TS unavailable/failed

LUNG_LABELS_TS = [
    'lung_upper_lobe_left',
    'lung_lower_lobe_left',
    'lung_upper_lobe_right',
    'lung_middle_lobe_right',
    'lung_lower_lobe_right',
]

def _dilate_mask(mask_2d, dilation_px):
    """Apply square dilation — same as dataset.py _get_lung_mask step 7."""
    if dilation_px <= 0:
        return mask_2d
    struct = np.ones((dilation_px * 2 + 1, dilation_px * 2 + 1), dtype=bool)
    return ndimage.binary_dilation(mask_2d.astype(bool), structure=struct).astype(np.uint8)


if not TS_AVAILABLE:
    print("TotalSegmentator not available — filling all TS masks with None.")
    ts_masks = [None] * len(samples)
else:
    try:
        import nibabel as nib
    except ImportError:
        print("nibabel not installed: pip install nibabel")
        ts_masks = [None] * len(samples)
        TS_AVAILABLE = False

if TS_AVAILABLE:
    for s in samples:
        pid = s['patient_id']
        try:
            tmpdir    = tempfile.mkdtemp(prefix=f"ts_{pid}_")
            nifti_in  = os.path.join(tmpdir, 'ct.nii.gz')
            nifti_out = os.path.join(tmpdir, 'seg')

            vol     = s['volume'].astype(np.float32)
            nib_img = nib.Nifti1Image(vol, affine=np.eye(4))
            nib.save(nib_img, nifti_in)

            device = 'gpu' if torch.cuda.is_available() else 'cpu'
            print(f"  Running TotalSegmentator on {pid} (device={device}) ...")
            run_ts(nifti_in, nifti_out, task='total', device=device, quiet=True)

            combined_3d = None
            for label_name in LUNG_LABELS_TS:
                seg_path = os.path.join(nifti_out, f'{label_name}.nii.gz')
                if os.path.exists(seg_path):
                    seg_data    = nib.load(seg_path).get_fdata()
                    combined_3d = seg_data if combined_3d is None else (combined_3d + seg_data)

            if combined_3d is None:
                raise RuntimeError("No lung lobe files found in TS output.")

            combined_3d = (combined_3d > 0).astype(np.uint8)
            k           = s['slice_idx']
            slice_mask  = combined_3d[:, :, k]

            # [ORACLE-CT, DAHAL ET AL. 2026] Mask dilation to capture boundary findings
            slice_mask = _dilate_mask(slice_mask, DILATION_PX)

            ts_masks.append(slice_mask)
            cov = slice_mask.mean() * 100
            print(f"    {pid}  TS mask coverage: {cov:.1f}%")

        except Exception as e:
            print(f"  WARNING — TotalSegmentator failed for {pid}: {e}")
            ts_masks.append(None)
        finally:
            shutil.rmtree(tmpdir, ignore_errors=True)

print(f"\nTS masks: {sum(m is not None for m in ts_masks)}/{len(ts_masks)} succeeded.")

## Cell 5 — Side-by-side visual comparison

Each row shows one nodule slice:
- **Col 1**: raw CT (greyscale, HU windowed −1000 to 400)
- **Col 2**: HU threshold mask overlay (green)
- **Col 3**: TotalSegmentator mask overlay (blue) or failure note
- **Col 4**: difference map — red = HU only, blue = TS only, grey = both agree

Saved to `results/mask_comparison.png`.

In [ ]:
HU_MIN, HU_MAX = -1000, 400

def _window(slice_2d):
    s = np.clip(slice_2d, HU_MIN, HU_MAX)
    return (s - HU_MIN) / (HU_MAX - HU_MIN)

n    = len(samples)
fig, axes = plt.subplots(n, 4, figsize=(16, 4 * n))

if n == 1:
    axes = axes[np.newaxis, :]

col_titles = ['CT slice', 'HU mask (green)', 'TS mask (blue)', 'Difference']
for j, title in enumerate(col_titles):
    axes[0, j].set_title(title, fontsize=11, fontweight='bold')

for i, (s, hu_mask, ts_mask) in enumerate(zip(samples, hu_masks, ts_masks)):
    ct_norm = _window(s['slice_2d'])
    pid     = s['patient_id']

    # Col 0 — raw CT
    axes[i, 0].imshow(ct_norm, cmap='gray', vmin=0, vmax=1)
    axes[i, 0].set_ylabel(pid, fontsize=8, rotation=0, labelpad=60, va='center')

    # Col 1 — HU mask overlay
    axes[i, 1].imshow(ct_norm, cmap='gray', vmin=0, vmax=1)
    overlay_hu = np.zeros((*hu_mask.shape, 4))
    overlay_hu[hu_mask > 0] = [0, 1, 0, 0.4]
    axes[i, 1].imshow(overlay_hu)

    # Col 2 — TS mask overlay (or failure note)
    axes[i, 2].imshow(ct_norm, cmap='gray', vmin=0, vmax=1)
    if ts_mask is not None:
        overlay_ts = np.zeros((*ts_mask.shape, 4))
        overlay_ts[ts_mask > 0] = [0, 0.4, 1, 0.4]
        axes[i, 2].imshow(overlay_ts)
    else:
        axes[i, 2].text(
            0.5, 0.5, 'TotalSegmentator\nnot available',
            ha='center', va='center', transform=axes[i, 2].transAxes,
            fontsize=9, color='white',
            bbox=dict(facecolor='black', alpha=0.6, boxstyle='round')
        )

    # Col 3 — Difference map
    axes[i, 3].imshow(ct_norm, cmap='gray', vmin=0, vmax=1)
    if ts_mask is not None:
        hu_b = hu_mask.astype(bool)
        ts_b = ts_mask.astype(bool)
        diff = np.zeros((*hu_b.shape, 4))
        diff[hu_b & ts_b]  = [0.5, 0.5, 0.5, 0.35]
        diff[hu_b & ~ts_b] = [1, 0, 0, 0.6]
        diff[~hu_b & ts_b] = [0, 0.4, 1, 0.6]
        axes[i, 3].imshow(diff)
        if i == 0:
            legend_patches = [
                mpatches.Patch(color=(0.5, 0.5, 0.5), label='Both agree'),
                mpatches.Patch(color='red',            label='HU only'),
                mpatches.Patch(color=(0, 0.4, 1),      label='TS only'),
            ]
            axes[i, 3].legend(handles=legend_patches, loc='lower right',
                              fontsize=7, framealpha=0.7)
    else:
        axes[i, 3].text(
            0.5, 0.5, 'N/A', ha='center', va='center',
            transform=axes[i, 3].transAxes, fontsize=12, color='white'
        )

    for ax in axes[i]:
        ax.axis('off')

plt.suptitle(
    'Lung Mask Comparison: HU Threshold vs TotalSegmentator\n'
    'Green = HU mask  |  Blue = TS mask  |  Difference: red=HU only, blue=TS only',
    fontsize=12, y=1.01
)
plt.tight_layout()

out_path = os.path.join(RESULTS_DIR, 'mask_comparison.png')
plt.savefig(out_path, dpi=120, bbox_inches='tight')
print(f"Figure saved to: {out_path}")
plt.show()

## Cell 6 — Quantitative comparison

For each sample where both masks are available, we compute:
- **HU coverage %**: fraction of slice pixels inside the HU lung mask
- **TS coverage %**: fraction of slice pixels inside the TotalSegmentator lung mask
- **IoU (HU vs TS)**: intersection-over-union between the two masks

High IoU means the two methods agree — validating that the physics-based HU threshold
is a reasonable, annotation-free approximation of a pretrained deep segmentor.

In [ ]:
def iou(mask_a, mask_b):
    a            = mask_a.astype(bool)
    b            = mask_b.astype(bool)
    intersection = (a & b).sum()
    union        = (a | b).sum()
    return intersection / union if union > 0 else float('nan')


header = f"{'Patient ID':<30} {'HU cov%':>10} {'TS cov%':>10} {'IoU HU vs TS':>14}"
sep    = '-' * len(header)
print(header)
print(sep)

iou_values = []

for s, hu_mask, ts_mask in zip(samples, hu_masks, ts_masks):
    pid    = s['patient_id']
    hu_cov = hu_mask.mean() * 100

    if ts_mask is not None:
        ts_cov  = ts_mask.mean() * 100
        iou_val = iou(hu_mask, ts_mask)
        iou_values.append(iou_val)
        print(f"{pid:<30} {hu_cov:>10.1f} {ts_cov:>10.1f} {iou_val:>14.4f}")
    else:
        print(f"{pid:<30} {hu_cov:>10.1f} {'N/A':>10} {'N/A':>14}")

print(sep)

if iou_values:
    mean_iou = np.mean(iou_values)
    print(f"\nMean IoU (HU vs TS): {mean_iou:.4f}  (n={len(iou_values)} paired samples)")
    if mean_iou >= 0.7:
        print("Interpretation: IoU >= 0.7 — HU threshold is a strong approximation of TotalSegmentator.")
    elif mean_iou >= 0.5:
        print("Interpretation: IoU 0.5–0.7 — HU threshold is a reasonable approximation with some discrepancy.")
    else:
        print("Interpretation: IoU < 0.5 — notable disagreement; inspect difference maps in Cell 5.")
else:
    print("\nNo paired samples — TotalSegmentator unavailable. HU coverage stats only:")
    covs = [hu_masks[i].mean() * 100 for i in range(len(samples))]
    print(f"  Mean HU coverage: {np.mean(covs):.1f}%  |  Std: {np.std(covs):.1f}%")

## Cell 7 — Summary and dissertation justification

### What this comparison shows

The two methods generate lung foreground masks using fundamentally different mechanisms:

| Property | HU Threshold (this dissertation) | TotalSegmentator (ORACLE-CT 2026) |
|----------|-----------------------------------|-----------------------------------|
| Mechanism | Physics: lung air < −500 HU | Deep learning: pretrained nnU-Net |
| Requires annotations | No | No (pretrained) |
| Requires GPU | No | Recommended |
| Requires pretrained model | No | Yes (~2 GB download) |
| Works on any CT scanner | Yes (HU is absolute) | Yes (generalises by design) |
| Speed per volume | ~0.1 s | ~30–120 s (CPU) |
| Novel as a training signal | **Yes** | No (used in ORACLE-CT) |

### Which mask covers more of the lung?

TotalSegmentator typically captures the full lung parenchyma (including the pleural lining)
because it was trained to do so. The HU threshold captures the air-filled interior of the
lung, then expands outward with 3-voxel dilation to include pleural and subpleural regions.
The net coverage is usually comparable, with HU sometimes slightly under-covering dense
consolidations (which are not air-dark) and TotalSegmentator slightly over-covering near
the pleural boundary.

### Is HU threshold a reasonable approximation?

If the IoU between the two methods is above ~0.6, this demonstrates that the physics-based
threshold and a state-of-the-art deep segmentor identify largely the same spatial region.
This validates the core assumption of the dissertation: a zero-annotation, zero-pretrained-model
mask is sufficient as a spatial training constraint for Grad-CAM attention regularisation.

### Justification for using HU threshold in this dissertation

1. **Zero annotations**: no manual lung masks, no manual nodule contours used at training time.
2. **Zero pretrained segmentors**: the mask is derived from physics (Hounsfield Units are an
   absolute physical quantity, not a learned representation), making it fully transferable
   to any CT scanner calibrated in HU.
3. **Novelty**: prior work (Zhang et al. 2022) uses a trained SegNet; ORACLE-CT (Dahal et al.
   2026) uses TotalSegmentator. Using HU thresholding *as a Grad-CAM training signal* —
   rather than merely for preprocessing — is the novel methodological contribution.
4. **Reproducibility**: any researcher with raw LIDC-IDRI DICOM files and the single
   threshold constant (−500 HU) can exactly reproduce the masks. No model weights needed.
5. **Speed**: generating all masks takes seconds on CPU, making the method accessible
   without specialised hardware.